# MEMS Release 외부 검산
공개 PLATES 사례에서 공급 지표와 두 홀층 좌표를 확인합니다.
**런타임 → 모두 실행**을 누르면 배포본의 입력과 출력을 내려받아 SciPy와 KLayout Python API로 검산합니다.
웹의 저장값과 새 결과를 비교하며, Windows 로컬 검산과 동일한 Python 소스와 보고서 형식을 사용합니다.
모델의 이산 방정식과 GDS 형상을 검산하는 범위입니다. 제조 규칙 또는 물리적 릴리스 시간의 확인은 별도 절차입니다.


In [ ]:
from hashlib import sha256
from pathlib import Path
from tempfile import mkdtemp
from urllib.request import urlopen

case_url = "https://praioe.github.io/mems-release-demo/release-example/"
work = Path(mkdtemp(prefix="mems-release-audit-"))
case_dir = work / "case"
case_dir.mkdir()
expected_hashes = {
    "source.gds": "811b0676de1588d0649422adb7036a3e007d67bfd13c74dfe1b8d223eea4ced1",
    "output.gds": "0552e001b5f93c78d2cfaa7fcf28211d1ffdb95c1f192ffb99a75b6b49a2b3d3",
    "replay.json": "04d603399b3b8691eeffba921342885edac4360a5715e69369ffd2a57b5934a0",
    "klayout-report.json": "36c18dd168fd9c528d84b9d1381cc41be609422080ef51211405ea427004a975"
}
for name, expected in expected_hashes.items():
    with urlopen(case_url + name, timeout=30) as stream:
        data = stream.read()
    if sha256(data).hexdigest() != expected:
        raise ValueError(f"Published asset does not match this notebook: {name}")
    (case_dir / name).write_bytes(data)
print("Verified four published inputs, including both GDS files.")


In [ ]:
import os
import subprocess
import sys

requirements = work / "requirements.txt"
requirements.write_text("scipy==1.18.0\nklayout==0.30.14\n", encoding="utf-8")
packages = work / "packages"
pip = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--disable-pip-version-check",
                      "--only-binary=:all:", "--target", str(packages), "-r", str(requirements)],
                     capture_output=True, text=True)
if pip.returncode:
    print(pip.stdout + pip.stderr)
pip.check_returncode()
python_env = {**os.environ, "PYTHONPATH": str(packages), "PYTHONNOUSERSITE": "1"}


In [ ]:
import json

verifier = work / "crosscheck.py"
verifier.write_text("\"\"\"Check the published MEMS specimen with SciPy and the standalone KLayout API.\"\"\"\nfrom collections import Counter\nimport argparse\nfrom hashlib import sha256\nimport json\nfrom pathlib import Path\nfrom time import perf_counter\n\nimport klayout.db as kdb\nfrom importlib.metadata import version\nimport numpy as np\nfrom scipy.sparse import coo_matrix\nfrom scipy.sparse.linalg import spsolve\n\nFIELD_TOLERANCE = 1e-8\nRESIDUAL_TOLERANCE = 1e-10\n\n\ndef solve(grid, holes, size, settings):\n    cells = grid[\"cells\"]\n    step = grid[\"step\"]\n    coordinates = np.array([(cell[\"x\"], cell[\"y\"]) for cell in cells])\n    centers = np.array([(hole[\"x\"], hole[\"y\"]) for hole in holes]).reshape(-1, 2)\n    half = np.array([hole.get(\"lowerSize\", size) for hole in holes])[:, None] / 2\n    widths = np.maximum(\n        0,\n        np.minimum(coordinates[:, None] + step / 2, centers + half)\n        - np.maximum(coordinates[:, None] - step / 2, centers - half),\n    )\n    covered_area = np.prod(widths, axis=2).sum(axis=1)\n    rhs = np.array([2 * cell[\"open\"] for cell in cells], dtype=float)\n    rhs += settings[\"exchange\"] * covered_area / step**2\n    beta = (step * grid[\"dbu\"] / settings[\"diffusionLengthUm\"]) ** 2\n    diagonal = np.array([len(cell[\"neighbors\"]) for cell in cells]) + rhs + beta\n    neighbors = [(i, j) for i, cell in enumerate(cells) for j in cell[\"neighbors\"]]\n    rows, cols, data = list(range(len(cells))), list(range(len(cells))), list(diagonal)\n    for first, second in neighbors:\n        rows.append(first)\n        cols.append(second)\n        data.append(-1)\n    matrix = coo_matrix((data, (rows, cols)), shape=(len(cells), len(cells))).tocsc()\n    field = spsolve(matrix, rhs, use_umfpack=False)\n    if not np.isfinite(field).all():\n        raise ValueError(\"The sparse solver returned a non-finite field.\")\n    return field, float(np.abs(matrix @ field - rhs).max())\n\n\ndef check_field(grid, holes, size, settings, expected):\n    field, residual = solve(grid, holes, size, settings)\n    if field.shape != np.asarray(expected[\"values\"]).shape:\n        raise ValueError(\"Reference field length differs from the recorded grid.\")\n    objective = float(np.mean((1 - field) ** 2))\n    field_error = float(np.max(np.abs(field - expected[\"values\"])))\n    objective_error = abs(objective - expected[\"objective\"])\n    return {\n        \"objective\": objective,\n        \"referenceObjective\": expected[\"objective\"],\n        \"worstDeficit\": float(1 - field.min()),\n        \"maxFieldDifference\": field_error,\n        \"objectiveDifference\": objective_error,\n        \"maxEquationResidual\": residual,\n        \"passed\": max(field_error, objective_error) <= FIELD_TOLERANCE\n        and residual <= RESIDUAL_TOLERANCE,\n    }\n\n\ndef check_gds(folder, design, proof, expected):\n    layout = kdb.Layout()\n    layout.read(str(folder / \"output.gds\"))\n    cell = layout.cell(expected[\"workCell\"])\n    if cell is None:\n        raise ValueError(\"The work cell is absent from the output GDS.\")\n    layers = []\n    for label, size_key in zip(design[\"holeLayers\"], [\"lowerSize\", \"upperSize\"]):\n        layer, datatype = map(int, label.split(\"/\"))\n        index = layout.find_layer(layer, datatype)\n        boxes = Counter()\n        non_rectangles = 0\n        for shape in ([] if index is None or index < 0 else cell.shapes(index).each()):\n            if shape.is_box():\n                box = shape.box\n            elif shape.is_polygon() and shape.polygon.is_box():\n                box = shape.bbox()\n            else:\n                non_rectangles += 1\n                continue\n            boxes[box.left + box.right, box.bottom + box.top, box.width(), box.height()] += 1\n        planned = Counter()\n        for hole in proof[\"chosen\"]:\n            size = hole.get(size_key, design[\"rules\"][size_key])\n            planned[2 * hole[\"x\"], 2 * hole[\"y\"], size, size] += 1\n        layers.append({\n            \"layer\": label,\n            \"expected\": sum(planned.values()),\n            \"found\": sum(boxes.values()) + non_rectangles,\n            \"missing\": sum((planned - boxes).values()),\n            \"unexpected\": sum((boxes - planned).values()) + non_rectangles,\n            \"passed\": planned == boxes and non_rectangles == 0,\n        })\n    source_hash = sha256((folder / \"source.gds\").read_bytes()).hexdigest()\n    output_hash = sha256((folder / \"output.gds\").read_bytes()).hexdigest()\n    hashes_match = source_hash == expected[\"sourceSha256\"] and output_hash == expected[\"outputSha256\"]\n    dbu_matches = abs(layout.dbu - proof[\"grid\"][\"dbu\"]) < 1e-12\n    return {\n        \"passed\": hashes_match and dbu_matches and all(row[\"passed\"] for row in layers),\n        \"workCell\": cell.name,\n        \"sourceSha256\": source_hash,\n        \"outputSha256\": output_hash,\n        \"hashesMatch\": hashes_match,\n        \"dbuMatches\": dbu_matches,\n        \"layers\": layers,\n    }\n\n\ndef audit(folder):\n    start = perf_counter()\n    replay = json.loads((folder / \"replay.json\").read_text(encoding=\"utf-8\"))\n    design, proof = replay[\"input\"], replay[\"proof\"]\n    expected_gds = json.loads((folder / \"klayout-report.json\").read_text(encoding=\"utf-8\"))\n    if replay[\"report\"] != expected_gds:\n        raise ValueError(\"The replay and the published KLayout report differ.\")\n    size = design[\"rules\"][\"lowerSize\"]\n    grid, settings = proof[\"grid\"], proof[\"settings\"]\n    before = check_field(grid, proof[\"baseline\"], size, settings, proof[\"before\"])\n    after = check_field(grid, proof[\"chosen\"], size, settings, proof[\"after\"])\n    gds = check_gds(folder, design, proof, expected_gds)\n    return {\n        \"schema\": \"mems-release-portable-audit-v1\",\n        \"passed\": before[\"passed\"] and after[\"passed\"] and gds[\"passed\"],\n        \"scope\": [\"discrete-model-field\", \"gds-layer-coordinates-and-sizes\", \"gds-sha256\"],\n        \"solver\": \"SciPy spsolve (SuperLU)\",\n        \"versions\": {name: version(name) for name in (\"numpy\", \"scipy\", \"klayout\")},\n        \"cells\": len(grid[\"cells\"]),\n        \"diffusionLengthUm\": settings[\"diffusionLengthUm\"],\n        \"fieldTolerance\": FIELD_TOLERANCE,\n        \"residualTolerance\": RESIDUAL_TOLERANCE,\n        \"before\": before,\n        \"after\": after,\n        \"gds\": gds,\n        \"elapsedMs\": round((perf_counter() - start) * 1000),\n    }\n\n\nif __name__ == \"__main__\":\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\"case_directory\", type=Path)\n    parser.add_argument(\"--report\", type=Path, required=True)\n    arguments = parser.parse_args()\n    result = audit(arguments.case_directory)\n    arguments.report.parent.mkdir(parents=True, exist_ok=True)\n    arguments.report.write_text(json.dumps(result, indent=2) + \"\\n\", encoding=\"utf-8\")\n    print(\n        f\"{'PASS' if result['passed'] else 'FAIL'}: objective \"\n        f\"{result['before']['objective']:.8f} -> {result['after']['objective']:.8f}; \"\n        f\"GDS {'matches' if result['gds']['passed'] else 'differs'}.\"\n    )\n    raise SystemExit(0 if result[\"passed\"] else 2)\n", encoding="utf-8")
report_path = work / "crosscheck.json"
execution = subprocess.run([sys.executable, "-B", str(verifier), str(case_dir),
                            "--report", str(report_path)], env=python_env,
                           capture_output=True, text=True)
print(execution.stdout + execution.stderr)
if report_path.exists():
    report = json.loads(report_path.read_text(encoding="utf-8"))
    print(json.dumps(report, indent=2, ensure_ascii=False))
execution.check_returncode()


보고서를 보관하려면 아래의 JSON 내려받기 버튼을 누르세요.


In [ ]:
from IPython.display import display
from google.colab import files
from ipywidgets import Button

download_button = Button(description="JSON 내려받기")
download_button.on_click(lambda _: files.download(str(report_path)))
display(download_button)
